# Fine-Tuning Llama 3.1 8B - Version 9 Complete (Code + Assembly)

**V9 Improvements:**
- ✅ **Complete code examples** - 3,550 C/C++ code examples generated with Grok
- ✅ **Compiler-verified code** - All code successfully compiled to x86-64 assembly
- ✅ **Behavior group training** - Two distinct question patterns:
  - Explain group: explanation + code (NO assembly)
  - Assembly group: code + assembly (when "assembly" in question)
- ✅ **Language-specific** - C and C++ code labeled correctly (```c vs ```cpp)
- ✅ **~15,500 training examples** - 12,888 train + 2,606 validation
- ✅ **Deep technical focus** - From concepts to verified assembly output

**Training Configuration:**
- 4 epochs
- Checkpoint every 500 steps
- Merged model in training data directory

**Prerequisites:**
- GPU Runtime (A100 recommended, T4 works)
- HuggingFace token with Llama access
- fine_tuning_llama_v9_clean folder uploaded to Google Drive

**Estimated Time:**
- A100: ~2 hours (4 epochs, 15.5k examples)
- T4: ~8 hours

## Step 1: Verify GPU

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies

In [ ]:
# Install Unsloth - optimized for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    # A100 = Ampere (compute capability 8.0)
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    # Older GPUs (T4, etc)
    !pip install --no-deps xformers trl peft accelerate bitsandbytes

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 4: Load V9 Complete Dataset

In [ ]:
import json
from datasets import Dataset

# Load V9 Complete training data from Google Drive
# UPDATE THIS PATH to where you uploaded fine_tuning_llama_v9_clean/
with open('/content/drive/MyDrive/fine_tuning_llama_v9_clean/train_with_code.json', 'r') as f:
    train_data = json.load(f)

with open('/content/drive/MyDrive/fine_tuning_llama_v9_clean/val_with_code.json', 'r') as f:
    val_data = json.load(f)

print(f"✅ Loaded {len(train_data)} training examples (V9 Complete)")
print(f"✅ Loaded {len(val_data)} validation examples (V9 Complete)")
print(f"\n📊 V9 Complete Dataset Quality:")
print(f"   - 3,550 C/C++ code examples (compiler-verified)")
print(f"   - Behavior groups: Explain (code only) vs Assembly (code+asm)")
print(f"   - Language-specific: C and C++ labeled correctly")
print(f"   - Real x86-64 assembly from gcc -S -O0")
print(f"   - Training: 12,888 examples (6,114 original + 6,774 code)")
print(f"   - Validation: 2,606 examples (680 original + 1,926 code)")
print(f"   - Total: 15,494 examples")

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# Show a sample
print("\n📝 Sample conversation (first 1000 chars):")
sample = train_data[0]
for msg in sample['messages']:
    role = msg['role'].upper()
    content = msg['content'][:150] + ("..." if len(msg['content']) > 150 else "")
    print(f"{role}: {content}")

## Step 5: Load Llama 3.1 Model with QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch
import os

# Model configuration - V9: Handle code + assembly
max_seq_length = 3072  # For assembly listings and code examples
dtype = None
load_in_4bit = True

# Load HuggingFace token
# OPTION 1: Upload .env file to Drive with HUGGINGFACE_TOKEN=your_token
env_path = '/content/drive/MyDrive/.env'
if os.path.exists(env_path):
    with open(env_path, 'r') as f:
        for line in f:
            if line.startswith('HUGGINGFACE_TOKEN='):
                HF_TOKEN = line.strip().split('=', 1)[1]
                print("✅ Token loaded from .env file")
                break
else:
    # OPTION 2: Paste your token here (less secure)
    HF_TOKEN = "your_token_here"  # ← REPLACE THIS
    print("⚠️  Using hardcoded token")

print("🔄 Loading Llama 3.1 8B model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = HF_TOKEN
)

print("✅ Model loaded successfully!")
print(f"📊 Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 6: Configure LoRA Adapters

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ LoRA adapters added!")
print(f"📊 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 7: Configure Chat Template & Formatting

In [ ]:
# Set Llama 3.1 chat template
tokenizer.chat_template = """{% if messages[0]['role'] == 'system' %}{% set system_message = messages[0]['content'] %}{% set loop_messages = messages[1:] %}{% else %}{% set loop_messages = messages %}{% endif %}{% if system_message is defined %}{{ '<|start_header_id|>system<|end_header_id|>\n\n' + system_message + '<|eot_id|>' }}{% endif %}{% for message in loop_messages %}{{ '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"""

print("✅ Chat template configured")

# Define formatting function - FIXED for SFTTrainer
def formatting_prompts_func(examples):
    """
    Format conversations with Llama 3.1 template and EOS token
    Returns dict with 'text' key for SFTTrainer
    """
    messages_list = examples["messages"]
    texts = []

    # messages_list is always a list of conversations (batch processing)
    for conversation in messages_list:
        if conversation is None or not isinstance(conversation, list):
            continue

        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False
        )

        # Ensure EOS token
        if not text.endswith(tokenizer.eos_token):
            text = text + tokenizer.eos_token

        texts.append(text)

    return {"text": texts}

print("✅ Formatting function defined")

## Step 8: Configure Training (V9 - 4 Epochs, Checkpoint every 500 steps)

In [ ]:
from transformers import TrainingArguments

# V9: 4 epochs, checkpoint every 500 steps
training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/fine_tuning_llama_v9_clean/checkpoints",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 100,
    num_train_epochs = 4,  # 4 epochs
    learning_rate = 2e-4,
    fp16 = False,
    bf16 = True,
    logging_steps = 20,
    optim = "adamw_torch_fused",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    
    # Checkpointing strategy - every 500 steps
    save_strategy = "steps",
    save_steps = 500,
    save_total_limit = 5,  # Keep last 5 checkpoints
    
    eval_strategy = "steps",
    eval_steps = 500,
    load_best_model_at_end = True,
    report_to = "none",
)

print("✅ V9 Training configuration ready!")
print(f"📊 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
total_steps = len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs
print(f"📊 Total training steps: ~{total_steps}")
print(f"💾 Checkpoints every {training_args.save_steps} steps (max {training_args.save_total_limit} kept)")
print(f"🔄 Number of epochs: {training_args.num_train_epochs}")

## Step 9: Create Trainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_prompts_func,
    max_seq_length=max_seq_length,
    args=training_args,
)

print("\n" + "="*70)
print("✅ TRAINER READY - VERSION 9 COMPLETE!")
print("="*70)
print(f"📊 Training examples: {len(train_dataset):,}")
print(f"📊 Validation examples: {len(val_dataset):,}")
print(f"🔄 Epochs: {training_args.num_train_epochs}")
print(f"💾 Checkpoints: Every 500 steps")
print(f"🎯 Focus: C/C++ code + verified assembly pairs")
print(f"🎯 Behavior groups: Explain (code) vs Assembly (code+asm)")
print(f"🔧 Max sequence: {max_seq_length} tokens")
print("="*70)
print("\n🚀 Ready to train! Run next cell.")

## Step 10: Start Fine-Tuning 🚀

**Configuration:**
- 4 epochs
- 15,494 examples (12,888 train + 2,606 val)
- Checkpoint every 500 steps
- Merged model will be saved in training data directory

**Estimated time:**
- A100: ~2 hours
- T4: ~8 hours

In [ ]:
import time

print("🚀 Starting V9 Complete training...")
print("="*70)
print(f"📊 Dataset: {len(train_dataset):,} training examples")
print(f"🔄 Epochs: {training_args.num_train_epochs}")
print(f"💾 Checkpoints: Every 500 steps")
print(f"🎯 Goal: C/C++ code + assembly understanding")
print(f"🎯 Behavior: Context-aware (assembly only when asked)")
print("="*70)
print("\nTraining in progress...\n")

start_time = time.time()
trainer_stats = trainer.train()
elapsed_time = time.time() - start_time

print("\n" + "="*70)
print("✅ V9 COMPLETE TRAINING DONE!")
print("="*70)
print(f"📊 Final Training Loss: {trainer_stats.training_loss:.4f}")
print(f"⏱️  Total Training Time: {elapsed_time/3600:.2f} hours ({elapsed_time/60:.0f} minutes)")
print(f"🔢 Total Steps: {trainer_stats.global_step}")
print(f"💾 Checkpoints saved to: {training_args.output_dir}")
print("="*70)

## Step 11: Find Latest Checkpoint

In [ ]:
import os

checkpoint_base = "/content/drive/MyDrive/fine_tuning_llama_v9_clean/checkpoints"

if os.path.exists(checkpoint_base):
    checkpoints = sorted([
        d for d in os.listdir(checkpoint_base)
        if d.startswith("checkpoint-")
    ], key=lambda x: int(x.split("-")[1]))
    
    print("Checkpoints found:")
    for cp in checkpoints[-5:]:  # Show last 5
        print(f"  - {cp}")
    
    latest = checkpoints[-1]
    latest_checkpoint = f"{checkpoint_base}/{latest}"
    print(f"\n🎯 Latest: {latest}")
    print(f"📍 Path: {latest_checkpoint}")
else:
    print(f"❌ Checkpoint directory not found: {checkpoint_base}")
    latest_checkpoint = None

## Step 12: Merge LoRA Adapter and Save Final Model

In [ ]:
from unsloth import FastLanguageModel
import os

if latest_checkpoint:
    print("🔄 Loading latest checkpoint...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=latest_checkpoint,
        max_seq_length=3072,
        dtype=None,
        load_in_4bit=True,
    )
    
    print("✅ Checkpoint loaded")
    print("🔄 Merging LoRA adapter with base model...")
    print("   (This takes 2-3 minutes...)")
    
    # Merge and save in the training data directory
    output_dir = "/content/drive/MyDrive/fine_tuning_llama_v9_clean/model_v9_merged"
    model.save_pretrained_merged(
        output_dir,
        tokenizer,
        save_method="merged_16bit",
    )
    
    print("")
    print("="*70)
    print("✅ SUCCESS! V9 Complete Model Merged!")
    print("="*70)
    print(f"")
    print(f"📍 Location: {output_dir}")
    print(f"📦 Format: Merged 16-bit model (HuggingFace format)")
    print(f"")
    print(f"🎯 Model Features:")
    print(f"   - 15,494 training examples")
    print(f"   - 4 epochs")
    print(f"   - Behavior groups: Explain vs Assembly")
    print(f"   - Language-specific: C and C++")
    print(f"   - Compiler-verified code + real assembly")
    print(f"")
    print(f"✅ Model ready for download and deployment!")
    print("="*70)
else:
    print("❌ No checkpoint found. Training may have failed.")

## Step 13: Test Your V9 Complete Model

In [ ]:
from unsloth import FastLanguageModel

output_dir = "/content/drive/MyDrive/fine_tuning_llama_v9_clean/model_v9_merged"

if os.path.exists(output_dir):
    # Load the final merged model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=output_dir,
        max_seq_length=3072,
        dtype=None,
        load_in_4bit=True,
    )
    
    FastLanguageModel.for_inference(model)
    
    print("✅ V9 Complete Model loaded and ready for inference!\n")
    
    def ask_model(question, max_tokens=1024):
        """Ask the V9 Complete model a question"""
        messages = [
            {
                "role": "system",
                "content": "You are a C/C++ systems programming expert. Provide precise, technically accurate explanations based on authoritative sources (K&R C, C++ Standard, C++ Primer, CSAPP). Focus on implementation details, compiler behavior, and standard-defined semantics. Avoid speculation or creative interpretation."
            },
            {
                "role": "user",
                "content": question
            }
        ]
        
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
        )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract assistant response
        assistant_marker = "<|start_header_id|>assistant<|end_header_id|>"
        if assistant_marker in response:
            response = response.split(assistant_marker)[-1].strip()
        
        return response
    
    # Test V9 knowledge
    test_questions = [
        "Explain RAII in C++",
        "Show me RAII in C++ and assembly",
        "What is virtual inheritance?",
        "Show me virtual inheritance in C++ and assembly",
    ]
    
    for q in test_questions:
        print("\n" + "="*70)
        print("Q:", q)
        print("="*70)
        answer = ask_model(q)
        print(answer[:600] + "..." if len(answer) > 600 else answer)
else:
    print(f"❌ Model not found at {output_dir}")

## 🎉 V9 Complete Training Done!

### Dataset Summary:
- **15,494 total examples** (12,888 train + 2,606 validation)
- **3,550 code examples** (C and C++)
- **1,609 with assembly** (compiler-verified, -O0)
- **Two behavior groups:**
  - Explain group: explanation + code (NO assembly)
  - Assembly group: code + assembly (when "assembly" mentioned)

### Training Configuration:
- **Model**: Llama 3.1 8B
- **Epochs**: 4
- **Checkpoints**: Every 500 steps
- **Training time**: ~2 hours (A100) or ~8 hours (T4)
- **Merged model location**: `/content/drive/MyDrive/fine_tuning_llama_v9_clean/model_v9_merged`

### Next Steps:
1. ✅ Download merged model from Drive (~16GB)
2. Convert to GGUF using llama.cpp (optional for local inference)
3. Test on C/C++ code + assembly queries
4. Deploy as needed

### Model Capabilities:
- **Contextual responses**: Assembly only when asked
- **Language-aware**: Correctly handles C and C++
- **Code + assembly**: Real compiler output pairs
- **Deep technical**: Concepts to implementation details

**Ready for deployment! 🚀**